In [ ]:
import pickle
import glob
import random
import h5py
import numpy as np
import os

from scipy.special import eval_hermite

import scfitpy
from scfitpy.image_processing import (
    apply_sato_filter,
    find_contours,
    assign_contours,
    determine_regions,
    determine_peak_positions,
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
# Configure matplotlib 
plt.rcParams["font.size"] = 25
plt.rcParams["mathtext.fontset"] = "stix"

In [ ]:
# Load test spectrum
_dat = np.load("./assets/cwSpec_currentSweep_202407011423.npz")
current = _dat["current"]  # mA
freq = _dat["freq"]  # GHz

mag1 = _dat["mag1"].T  # 2d-spectrum date, dims=[freq, current]

print(f"{mag1.shape=} {current.shape=} {freq.shape=}")

In [ ]:
mag1

In [ ]:
# Check the spectrum
fig, ax = plt.subplots(figsize=(8, 6))

X, Y = np.meshgrid(current, freq)
mappable = ax.pcolor(X, Y, mag1, cmap="summer")
cbar = plt.colorbar(mappable, ax=ax, format="%.2f")

plt.xlabel(r"Current [mA]")
plt.ylabel(r"$\omega_p\ $[GHz]")
plt.show()

In [ ]:
mag1_filtered = apply_sato_filter(
    mag1, 4, black_ridges=True, with_plot=True, filename_plot="./outs/sato.png"
)

In [ ]:
cont_list = find_contours(
    mag1_filtered, level=0.016, with_plot=True, filename_plot="./outs/cont.png"
)
print(f"# of contours: {len(cont_list)}")

In [ ]:
cont_dict = assign_contours(
    cont_list,
    dict(
        a=(12, 11), b=(17, 18), c=(16, 15)
    ), 
    with_plot=True,
    filename_plot="./outs/cont_assignments.png",
)

In [ ]:
cont_dict

In [ ]:
region_dict = determine_regions(mag1, cont_dict, 5, with_plot=True)

In [ ]:
region_dict

In [ ]:
peak_dict = determine_peak_positions(
    mag1_filtered,
    region_dict,
    xaxis=current,
    yaxis=freq,
    with_plot=True,
)

for kw, pos in peak_dict.items():
    print(f'band:{kw} --> (xval,yval)={pos}')

In [ ]:
with open('./outs/peak_dict.pkl', 'wb') as file:
    pickle.dump(peak_dict, file)